# Build retrieval indexes

**Goal:** build external document memory for RAG.

We build:

1. Dense FAISS index with `sentence-transformers/all-MiniLM-L6-v2`.
2. BM25 sparse index.
3. Retrieval-quality evaluation with Recall@k and SupportRecall@k.

This notebook does not use the language model yet. It isolates the retriever.

In [1]:
%pip install -r ../requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/polinakorobeinikova/IU/NLP/case-study/case-study-rag-factual-consistency


In [3]:
import json
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.config import PROCESSED_DIR, INDEX_DIR, EMBEDDING_MODEL
from src.data_utils import load_gold_doc_ids
from src.metrics import recall_at_k, support_recall_at_k, safe_mean
from src.retrieval import (
    build_dense_index, build_bm25_index,
    DenseRetriever, BM25Retriever, retrieved_doc_ids
)
from src.analysis_utils import file_size_mb, save_json

pd.set_option("display.max_colwidth", 160)

In [4]:
questions_df = pd.read_parquet(PROCESSED_DIR / "questions.parquet")
corpus_df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

print(questions_df.shape, corpus_df.shape)
display(questions_df.head(2))
display(corpus_df.head(2))

(5000, 7) (49774, 8)


,sample_id,question,answer,type,level,support_titles,gold_doc_ids
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,comparison,hard,"[""Ed Wood"", ""Scott Derrickson""]","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]"
1,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,bridge,hard,"[""Kiss and Tell (1945 film)"", ""Shirley Temple""]","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]"


,doc_id,sample_id,title,paragraph_idx,text,sentences_json,support_sentence_ids,is_supporting_doc
0,5a8b57f25542995d1e6f1371::0::Ed Wood (film),5a8b57f25542995d1e6f1371,Ed Wood (film),0,"Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood. T...","[""Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.""...",[],False
1,5a8b57f25542995d1e6f1371::1::Scott Derrickson,5a8b57f25542995d1e6f1371,Scott Derrickson,1,"Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. He lives in Los Angeles, California. He is best known for direct...","[""Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer."", "" He lives in Los Angeles, California."", "" He is best known fo...",[0],True


## Dense FAISS index

In [5]:
INDEX_DIR.mkdir(parents=True, exist_ok=True)

dense_index_path = INDEX_DIR / "dense_faiss.index"
embeddings_path = INDEX_DIR / "dense_embeddings.npy"
doc_ids_path = INDEX_DIR / "dense_doc_ids.json"

start = time.perf_counter()
index, embeddings = build_dense_index(
    corpus_df=corpus_df,
    model_name=EMBEDDING_MODEL,
    index_path=dense_index_path,
    embeddings_path=embeddings_path,
    doc_ids_path=doc_ids_path,
    batch_size=64,
    normalize_embeddings=True,
)
dense_build_time = time.perf_counter() - start

print("Embeddings:", embeddings.shape)
print("Dense index size MB:", file_size_mb(dense_index_path))
print("Embeddings size MB:", file_size_mb(embeddings_path))
print("Build time sec:", dense_build_time)

Batches:   0%|          | 0/778 [00:00<?, ?it/s]

Embeddings: (49774, 384)
Dense index size MB: 72.91117572784424
Embeddings size MB: 72.9112548828125
Build time sec: 198.91198320799958


## BM25 sparse index

In [6]:
bm25_path = INDEX_DIR / "bm25.pkl"

start = time.perf_counter()
bm25 = build_bm25_index(corpus_df, bm25_path)
bm25_build_time = time.perf_counter() - start

print("BM25 size MB:", file_size_mb(bm25_path))
print("Build time sec:", bm25_build_time)

BM25 size MB: 56.24495029449463
Build time sec: 2.246472791999622


## Evaluate retrieval quality

We measure whether the retriever returns at least one gold supporting paragraph and what fraction of gold supporting paragraphs appear in the top-k.

In [7]:
dense_retriever = DenseRetriever(dense_index_path, doc_ids_path, corpus_df, EMBEDDING_MODEL)
bm25_retriever = BM25Retriever(bm25_path, corpus_df)

EVAL_N = min(5000, len(questions_df))
TOP_K_VALUES = [1, 3, 5, 10]

eval_questions = questions_df.head(EVAL_N).copy()

In [8]:
def evaluate_retriever(name, retriever, questions):
    rows = []
    for _, row in tqdm(questions.iterrows(), total=len(questions), desc=name):
        gold_doc_ids = load_gold_doc_ids(row["gold_doc_ids"])
        chunks = retriever.retrieve(row["question"], top_k=max(TOP_K_VALUES))
        ret_doc_ids = retrieved_doc_ids(chunks)

        out = {
            "retriever": name,
            "sample_id": row["sample_id"],
            "question": row["question"],
            "answer": row["answer"],
            "retrieved_doc_ids": json.dumps(ret_doc_ids, ensure_ascii=False),
            "gold_doc_ids": json.dumps(gold_doc_ids, ensure_ascii=False),
        }

        for k in TOP_K_VALUES:
            out[f"hit_at_{k}"] = recall_at_k(ret_doc_ids, gold_doc_ids, k)
            out[f"support_recall_at_{k}"] = support_recall_at_k(ret_doc_ids, gold_doc_ids, k)

        rows.append(out)

    return pd.DataFrame(rows)

dense_eval = evaluate_retriever("dense", dense_retriever, eval_questions)
bm25_eval = evaluate_retriever("bm25", bm25_retriever, eval_questions)

retrieval_eval_df = pd.concat([dense_eval, bm25_eval], ignore_index=True)
retrieval_eval_df.head()

dense:   0%|          | 0/5000 [00:00<?, ?it/s]

bm25:   0%|          | 0/5000 [00:00<?, ?it/s]

,retriever,sample_id,question,answer,retrieved_doc_ids,gold_doc_ids,hit_at_1,support_recall_at_1,hit_at_3,support_recall_at_3,hit_at_5,support_recall_at_5,hit_at_10,support_recall_at_10
0,dense,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same nationality?,yes,"[""5a8b6d885542997f31a41d25::9::Ed Wood (film)"", ""5a8b57f25542995d1e6f1371::0::Ed Wood (film)"", ""5a84d2ea5542992a431d1aa8::0::Ed Wood"", ""5a8b57f25542995d1e6f...","[""5a8b57f25542995d1e6f1371::1::Scott Derrickson"", ""5a8b57f25542995d1e6f1371::4::Ed Wood""]",0.0,0.0,0.0,0.0,1.0,0.5,1.0,0.5
1,dense,5a8c7595554299585d9e36b6,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,"[""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)"", ""5ae1b1445542997283cd223d::7::A Kiss for Corliss"", ""5a8c7595554299585d9e36b6::5::A Kiss for Corli...","[""5a8c7595554299585d9e36b6::1::Shirley Temple"", ""5a8c7595554299585d9e36b6::6::Kiss and Tell (1945 film)""]",1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5
2,dense,5a85ea095542994775f606a8,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,"[""5abeebe25542993fe9a41d9c::6::First North Americans"", ""5a73024f5542991f9a20c5fc::5::Multiverse: Exploring Poul Anderson's Worlds"", ""5a85ea095542994775f606a...","[""5a85ea095542994775f606a8::2::The Hork-Bajir Chronicles"", ""5a85ea095542994775f606a8::8::Animorphs""]",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,dense,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,"[""5adbf0a255429947ff17385a::6::Esma Sultan Mansion"", ""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5a88a35c554299206df2b316::2::Abidin Mosque"", ""5adbf0a255...","[""5adbf0a255429947ff17385a::5::Laleli Mosque"", ""5adbf0a255429947ff17385a::6::Esma Sultan Mansion""]",1.0,0.5,1.0,1.0,1.0,1.0,1.0,1.0
4,dense,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City","[""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)"", ""5ade15f35542997545bbbe47::0::Manhattan Romance"", ""5ae0bae555429906c02dab08::9::Stone Stanley Entertai...","[""5a8e3ea95542995a26add48d::3::Adriana Trigiani"", ""5a8e3ea95542995a26add48d::9::Big Stone Gap (film)""]",1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5


In [9]:
summary_cols = [c for c in retrieval_eval_df.columns if c.startswith("hit_at_") or c.startswith("support_recall_at_")]
retrieval_summary = retrieval_eval_df.groupby("retriever")[summary_cols].mean().reset_index()
display(retrieval_summary)

retrieval_eval_df.to_csv(INDEX_DIR / "retrieval_eval.csv", index=False)
retrieval_summary.to_csv(INDEX_DIR / "retrieval_summary.csv", index=False)

build_summary = {
    "embedding_model": EMBEDDING_MODEL,
    "n_documents": len(corpus_df),
    "embedding_dim": int(np.load(embeddings_path, mmap_mode="r").shape[1]),
    "dense_index_size_mb": file_size_mb(dense_index_path),
    "dense_embeddings_size_mb": file_size_mb(embeddings_path),
    "bm25_size_mb": file_size_mb(bm25_path),
    "dense_build_time_sec": dense_build_time,
    "bm25_build_time_sec": bm25_build_time,
}

save_json(INDEX_DIR / "index_build_summary.json", build_summary)
build_summary

,retriever,hit_at_1,support_recall_at_1,hit_at_3,support_recall_at_3,hit_at_5,support_recall_at_5,hit_at_10,support_recall_at_10
0,bm25,0.6460,0.3230,0.8180,0.5193,0.8790,0.6002,0.9398,0.7083
1,dense,0.7078,0.3539,0.8662,0.5633,0.9084,0.6323,0.9406,0.7047


{'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'n_documents': 49774,
 'embedding_dim': 384,
 'dense_index_size_mb': 72.91117572784424,
 'dense_embeddings_size_mb': 72.9112548828125,
 'bm25_size_mb': 56.24495029449463,
 'dense_build_time_sec': 198.91198320799958,
 'bm25_build_time_sec': 2.246472791999622}

Dense retrieval outperforms BM25 on HotpotQA evidence retrieval at low k. At top-10, dense and BM25 are almost tied. `BM25 SupportRecall@10 = 0.7083` and `Dense SupportRecall@10 = 0.7047`. This supports using dense FAISS retrieval as the vanilla RAG baseline. However, supporting-fact recall remains lower than hit rate, showing that retrieving at least one relevant paragraph is easier than retrieving all evidence needed for multi-hop factual reasoning. Therefore, the advanced RAG stage is motivated by the need to improve evidence completeness, not just top-1 relevance.

BM25 is much cheaper to build: 2.35 sec vs 198.91 sec. Dense retrieval requires embedding all documents, so build time is much higher. However, the final memory overhead is still small for this corpus: around 72.9 MB for the FAISS index and 72.9 MB for stored embeddings.